# 02 · Data cleaning

**Project:** Banking Transaction Analytics (Berka `financial` data)
**Goal:** turn the raw tables into analysis-ready tables for MySQL and Power BI, **without deleting or changing any raw value**, and prove that totals before and after cleaning reconcile.

**What this notebook does**
1. Loads the raw CSV files (read-only) and records baseline totals.
2. Adds derived columns, labels and flags, following the decisions made after profiling (table below).
3. Rebuilds the other tables with readable names and privacy-safe client fields.
4. Reconciles row counts and totals before and after, and re-checks keys.
5. Writes a data quality report, a list of exceptions, and the clean files.

**Outputs**
- `data/processed/*.csv`: clean tables (**not committed**: `.gitignore` excludes `data/`).
- `data/processed/dq_exceptions.csv`: row-level exceptions (**not committed**).
- `reports/data_quality_report.csv` and `.md`, `reports/cleaning_reconciliation.csv`, `reports/cleaning_checks.csv`: counts and totals only (safe to commit).

## Decisions carried over from profiling

| Issue | What profiling showed | Decision |
|---|---|---|
| `operation` blank | Exactly the 183,114 interest (`UROK`) rows | Keep. `operation_clean = "NONE"`; interest rows get the category "Interest credited" |
| `k_symbol` has two kinds of "none" | 481,881 truly missing and 53,433 single spaces (spaces only on debits) | `k_symbol_clean = "NONE"` for both; `k_symbol_kind` records which |
| `bank` / `account` | A bank always comes with a real partner account (273,508 rows); `account = 0` on 21,881 cash and card withdrawals, never with a bank | `partner_account` is NULL when 0; `partner_account_was_zero` flag |
| Zero amounts | 162 rows, all interest | Keep; `is_zero_amount` flag |
| Negative balances | 2,999 rows in 288 accounts | Keep; `is_negative_balance` flag |
| `balance` column | `amount` and `balance` are whole numbers. Day by day, the chain holds within 1 on all but 66 account-days, and the worst difference is at most 5 | Keep raw values; add `balance_before` and a `balance_check` flag |
| Order within a day | `trans_id` is not chronological, so the order inside a day is unknown | No order is invented. Use `date`; `balance_before = balance - signed_amount` works per row |
| Identical-looking rows | 3,448 rows in 1,723 groups; 94.5% of groups have balances spaced by the amount (within 1) | Keep all rows; flag them; flag the few groups whose members share one balance |
| Loan status | Fully determined by "balance ever negative" and "end date reaches 1999" | Keep `status`; add labels and `is_finished`; **no predictive claims** |
| Exact birth date | Personal data | Export birth year, age and age band only |

Czech code meanings used for labels are **to be confirmed** against the original documentation.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)


def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find data/raw."""
    for folder in [start, *start.parents]:
        if (folder / "data" / "raw").exists():
            return folder
    raise FileNotFoundError("Could not find data/raw. Run scripts/download_berka.py first.")


ROOT = find_project_root(Path.cwd())
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
print("Project root:", ROOT)

Project root: c:\Users\acer\Documents\banking-transaction-analytics


In [2]:
checks = []
dq = []
exception_frames = []


def check(name, passed, detail=""):
    """Record one check. Any FAIL stops the notebook at the end."""
    passed = bool(passed)
    checks.append({"check": name, "status": "PASS" if passed else "FAIL", "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}" + (f"  ({detail})" if detail else ""))


def log_issue(table, column, issue, rows, total, handling):
    """Add one line to the data quality report."""
    dq.append(
        {
            "table": table,
            "column": column,
            "issue": issue,
            "rows_affected": int(rows),
            "share_pct": round(100 * rows / total, 2) if total else None,
            "handling": handling,
        }
    )


def add_exceptions(issue, table, keys, note):
    keys = list(keys)
    if keys:
        exception_frames.append(pd.DataFrame({"issue": issue, "table": table, "key": keys, "note": note}))

## 1. Load the raw data and record baseline totals

In [3]:
SMALL_TABLES = ["account", "card", "client", "disp", "district", "loan", "order"]
raw = {name: pd.read_csv(RAW / f"{name}.csv") for name in SMALL_TABLES}
for table, columns in {"account": ["date"], "card": ["issued"], "client": ["birth_date"], "loan": ["date"]}.items():
    for col in columns:
        raw[table][col] = pd.to_datetime(raw[table][col])

trans = pd.read_csv(
    RAW / "trans.csv",
    dtype={
        "trans_id": "int32",
        "account_id": "int32",
        "type": "category",
        "operation": "category",
        "k_symbol": "category",
        "bank": "category",
    },
    parse_dates=["date"],
    low_memory=False,  # one consistent type per category column
)
raw["trans"] = trans

FULL = len(trans) == 1_056_320  # the known size of the full dataset: regression checks only apply then
print(f"trans rows: {len(trans):,}  (full dataset: {FULL})")

baseline = {
    "rows": {name: len(df) for name, df in raw.items()},
    "amount_by_type": trans.groupby("type", observed=True)["amount"].sum().to_dict(),
    "rows_by_type": trans["type"].value_counts().to_dict(),
    "balance_sum": float(trans["balance"].sum()),
    "first_date": trans["date"].min(),
    "last_date": trans["date"].max(),
    "n_accounts": trans["account_id"].nunique(),
}
pd.Series({name: f"{n:,}" for name, n in baseline["rows"].items()}, name="raw rows").to_frame()

trans rows: 1,056,320  (full dataset: True)


,raw rows
account,"4,500"
card,892
client,"5,369"
disp,"5,369"
district,77
loan,682
order,"6,471"
trans,"1,056,320"


## 2. Transactions: amounts, labels and date parts
`PRIJEM` is a credit; `VYDAJ` and `VYBER` are debits. Profiling showed that this sign convention reproduces the balance chain.

In [4]:
tc = trans.copy()
tc["direction"] = np.where(tc["type"].eq("PRIJEM"), "credit", "debit")
tc["signed_amount"] = np.where(tc["direction"].eq("credit"), tc["amount"], -tc["amount"])
tc["balance_before"] = tc["balance"] - tc["signed_amount"]  # the balance just before this posting (no ordering needed)
tc["type_label"] = tc["type"].astype("object").map({"PRIJEM": "Credit", "VYDAJ": "Debit", "VYBER": "Cash withdrawal"})
tc["is_zero_amount"] = tc["amount"].eq(0)
tc["is_negative_balance"] = tc["balance"].lt(0)

tc["year"] = tc["date"].dt.year
tc["month"] = tc["date"].dt.month
tc["year_month"] = tc["date"].dt.strftime("%Y-%m")
tc["month_start"] = tc["date"].dt.to_period("M").dt.to_timestamp()

print("Zero-amount rows:", int(tc["is_zero_amount"].sum()), "| negative-balance rows:", int(tc["is_negative_balance"].sum()))

Zero-amount rows: 162 | negative-balance rows: 2999


## 3. Transaction codes: `operation`, `k_symbol` and a readable category
Meanings below come from the original documentation and are **to be confirmed**.

In [5]:
operation = tc["operation"].astype("object")
k_symbol = tc["k_symbol"].astype("object")

tc["k_symbol_kind"] = np.select([k_symbol.isna(), k_symbol.eq(" ")], ["missing", "space"], default="value")
tc["k_symbol_clean"] = k_symbol.where(tc["k_symbol_kind"].eq("value"), "NONE")
tc["operation_clean"] = operation.fillna("NONE")

K_MEANING = {
    "UROK": "Interest credited",
    "SANKC. UROK": "Penalty interest",
    "SLUZBY": "Service fees",
    "SIPO": "Household payment",
    "DUCHOD": "Pension credit",
    "POJISTNE": "Insurance payment",
    "UVER": "Loan payment",
}
OP_MEANING = {
    "VKLAD": "Cash deposit",
    "PREVOD Z UCTU": "Incoming transfer",
    "PREVOD NA UCET": "Outgoing transfer",
    "VYBER": "Cash withdrawal",
    "VYBER KARTOU": "Card withdrawal",
}
category = tc["k_symbol_clean"].map(K_MEANING)
category = category.fillna(operation.map(OP_MEANING)).fillna("Other")
tc["transaction_category"] = category

print("k_symbol kinds:")
display(tc["k_symbol_kind"].value_counts().rename("rows").to_frame())
print("Transaction categories:")
display(tc["transaction_category"].value_counts().rename("rows").to_frame())
check("Every transaction gets a category other than 'Other'", (tc["transaction_category"] != "Other").all(), f"{int((tc['transaction_category'] == 'Other').sum()):,} rows are 'Other'")

k_symbol kinds:


,rows
k_symbol_kind,
value,521006
missing,481881
space,53433


Transaction categories:


,rows
transaction_category,
Cash withdrawal,274675
Interest credited,183114
Cash deposit,156743
Service fees,155832
Household payment,118065
Outgoing transfer,60972
Incoming transfer,34888
Pension credit,30338
Insurance payment,18500


[PASS] Every transaction gets a category other than 'Other'  (0 rows are 'Other')


## 4. Counterparty (partner bank and account)
`account = 0` never occurs together with a bank, and only on cash and card withdrawals. We keep the raw zero as a flag and use NULL in the clean column.

In [6]:
tc["partner_bank"] = tc["bank"]
tc["partner_account_was_zero"] = tc["account"].eq(0)
tc["partner_account"] = tc["account"].where(tc["account"].ne(0)).astype("Int64")

has_counterparty = tc["partner_bank"].notna() & tc["partner_account"].notna()
print(f"Rows with a partner bank and account: {int(has_counterparty.sum()):,}")
print(f"Rows with partner_account_was_zero:   {int(tc['partner_account_was_zero'].sum()):,}")
check("A partner bank is present exactly when there is a real partner account", (tc["partner_bank"].notna() == tc["partner_account"].notna()).all())

no_counterparty_transfer = tc[tc["operation"].eq("PREVOD NA UCET") & tc["partner_bank"].isna() & tc["partner_account"].isna()]
print(f"Outgoing transfers without a counterparty: {len(no_counterparty_transfer):,}")

Rows with a partner bank and account: 273,508
Rows with partner_account_was_zero:   21,881
[PASS] A partner bank is present exactly when there is a real partner account
Outgoing transfers without a counterparty: 1


## 5. Identical-looking postings
Rows that match in every column except `trans_id` are **kept** and flagged. Profiling showed that in about 95% of these groups the balances are spaced by the amount, so the rows are real separate postings. Groups whose members share the same balance (and have a non-zero amount) get a separate flag for review.

In [7]:
DUP_COLS = ["account_id", "date", "type", "operation", "amount", "k_symbol", "bank", "account"]
tc["repeat_group_size"] = tc.groupby(DUP_COLS, observed=True, dropna=False)["trans_id"].transform("size").astype("int16")
tc["is_repeated_posting"] = tc["repeat_group_size"].gt(1)

repeated = tc[tc["is_repeated_posting"]]
same_balance = repeated.groupby(DUP_COLS + ["balance"], observed=True, dropna=False)["trans_id"].transform("size").gt(1) & repeated["amount"].gt(0)
tc["possible_phantom_copy"] = False
tc.loc[same_balance[same_balance].index, "possible_phantom_copy"] = True

n_repeated_groups = int(round((1 / repeated["repeat_group_size"]).sum()))
n_phantom_groups = int(round((1 / tc.loc[tc["possible_phantom_copy"], "repeat_group_size"]).sum())) if tc["possible_phantom_copy"].any() else 0
print(f"Identical-looking rows: {len(repeated):,} in {n_repeated_groups:,} groups")
print(f"Rows flagged possible_phantom_copy (same balance, amount > 0): {int(tc['possible_phantom_copy'].sum()):,} (about {n_phantom_groups:,} groups)")

Identical-looking rows: 3,448 in 1,723 groups
Rows flagged possible_phantom_copy (same balance, amount > 0): 8 (about 4 groups)


## 6. Balance check per account-day
For each account-day we ask: is there a balance of the previous day and a balance of this day such that `previous + net amount of the day = this balance`? The size of the smallest difference is stored. Profiling found it is at most 5 everywhere and 0 or 1 on almost all days, which fits rounding of whole-number amounts and balances.

`balance_check`: **exact** (up to 0.01), **rounding** (up to 2), **review** (above 2).

In [8]:
daily = tc.groupby(["account_id", "date"]).agg(net=("signed_amount", "sum"), rows=("trans_id", "size"))
days = daily.reset_index().sort_values(["account_id", "date"])
days["prev_date"] = days.groupby("account_id")["date"].shift(1)

ends = tc[["account_id", "date", "balance"]].rename(columns={"balance": "end_candidate"})
starts = tc[["account_id", "date", "balance"]].rename(columns={"date": "prev_date", "balance": "start_candidate"})
pairs = ends.merge(days[["account_id", "date", "prev_date", "net"]], on=["account_id", "date"])
pairs = pairs.merge(starts, on=["account_id", "prev_date"], how="left")
pairs["start_candidate"] = pairs["start_candidate"].where(pairs["prev_date"].notna(), 0)  # before the first posting the balance is 0
pairs["abs_resid"] = (pairs["start_candidate"] + pairs["net"] - pairs["end_candidate"]).abs()
day_resid = pairs.groupby(["account_id", "date"])["abs_resid"].min().rename("account_day_residual")
del pairs, ends, starts

tc = tc.join(day_resid, on=["account_id", "date"]).join(daily["rows"].rename("account_day_rows"), on=["account_id", "date"])
tc["balance_check"] = pd.cut(tc["account_day_residual"], [-0.001, 0.01, 2.0, np.inf], labels=["exact", "rounding", "review"])

day_level = pd.cut(day_resid, [-0.001, 0.01, 2.0, np.inf], labels=["exact", "rounding", "review"]).value_counts(sort=False)
print("Account-days by balance check:")
display(day_level.rename("account_days").to_frame())
print("Transactions by balance check:")
display(tc["balance_check"].value_counts(sort=False).rename("rows").to_frame())

review_days = day_resid[day_resid > 2].reset_index()
add_exceptions(
    "balance_check = review",
    "trans",
    (review_days["account_id"].astype(str) + "|" + review_days["date"].dt.strftime("%Y-%m-%d")),
    "account-day whose best-case balance difference is above 2",
)
check("No account-day has a balance difference above 5", day_resid.max() <= 5, f"largest difference {day_resid.max():,.2f}")

Account-days by balance check:


,account_days
account_day_residual,
exact,750640
rounding,64127
review,1


Transactions by balance check:


,rows
balance_check,
exact,928604
rounding,127709
review,7


[PASS] No account-day has a balance difference above 5  (largest difference 3.00)


## 7. The other tables
Readable column names for `district` (names to be confirmed), privacy-safe client fields, account attributes (one account has one owner), loan labels and card/order tidy-up.

In [9]:
# --- district: descriptive names ---
district_c = raw["district"].rename(
    columns={
        "A2": "district_name",
        "A3": "region",
        "A4": "population",
        "A5": "municipalities_under_500",
        "A6": "municipalities_500_1999",
        "A7": "municipalities_2000_9999",
        "A8": "municipalities_10000_plus",
        "A9": "cities",
        "A10": "urban_share_pct",
        "A11": "average_salary",
        "A12": "unemployment_rate_1995",
        "A13": "unemployment_rate_1996",
        "A14": "entrepreneurs_per_1000",
        "A15": "crimes_1995",
        "A16": "crimes_1996",
    }
)

# --- client: no exact birth date, only birth year, age and age band ---
REFERENCE_DATE = pd.Timestamp("1998-12-31")  # last day in the data
AGE_BINS = [-1, 17, 24, 34, 44, 54, 64, 200]
AGE_LABELS = ["under 18", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]


def age_on(dob, ref):
    """Completed years on the reference date (both are datetime Series)."""
    had_birthday = (ref.dt.month > dob.dt.month) | ((ref.dt.month == dob.dt.month) & (ref.dt.day >= dob.dt.day))
    return ref.dt.year - dob.dt.year - (~had_birthday).astype(int)


client = raw["client"].copy()
client["birth_year"] = client["birth_date"].dt.year
client["age_end_1998"] = age_on(client["birth_date"], pd.Series(REFERENCE_DATE, index=client.index))
client["age_band_end_1998"] = pd.cut(client["age_end_1998"], AGE_BINS, labels=AGE_LABELS)
client_c = client[["client_id", "gender", "birth_year", "age_end_1998", "age_band_end_1998", "district_id"]]

# --- account: owner attributes, products ---
disp = raw["disp"]
owner = disp[disp["type"] == "OWNER"][["account_id", "client_id"]].rename(columns={"client_id": "owner_client_id"})
n_disponents = disp[disp["type"] == "DISPONENT"].groupby("account_id").size().rename("n_disponents")
card_link = raw["card"].merge(disp[["disp_id", "account_id", "client_id"]], on="disp_id", how="left")

account = raw["account"].rename(columns={"date": "opened_date", "frequency": "frequency_code"})
account = account.merge(owner, on="account_id", how="left").merge(
    client[["client_id", "gender", "birth_date"]].rename(columns={"client_id": "owner_client_id", "gender": "owner_gender"}),
    on="owner_client_id",
    how="left",
)
account["owner_age_at_opening"] = age_on(account["birth_date"], account["opened_date"])
account["owner_age_band"] = pd.cut(account["owner_age_at_opening"], AGE_BINS, labels=AGE_LABELS)
account["opened_year"] = account["opened_date"].dt.year
account["frequency_label"] = account["frequency_code"].map(
    {"POPLATEK MESICNE": "Monthly", "POPLATEK TYDNE": "Weekly", "POPLATEK PO OBRATU": "After each transaction"}
)
account = account.join(n_disponents, on="account_id")
account["n_disponents"] = account["n_disponents"].fillna(0).astype(int)
account["has_loan"] = account["account_id"].isin(raw["loan"]["account_id"])
account["has_card"] = account["account_id"].isin(card_link["account_id"])
account_c = account[
    [
        "account_id", "district_id", "frequency_code", "frequency_label", "opened_date", "opened_year",
        "owner_client_id", "owner_gender", "owner_age_at_opening", "owner_age_band",
        "n_disponents", "has_loan", "has_card",
    ]
]

# --- loan: labels and end date ---
loan = raw["loan"].copy()
loan["end_date"] = pd.to_datetime([d + pd.DateOffset(months=int(m)) for d, m in zip(loan["date"], loan["duration"])])
loan["status_label"] = loan["status"].map(
    {"A": "Finished, paid", "B": "Finished, unpaid", "C": "Running, in order", "D": "Running, in debt"}
)
loan["is_finished"] = loan["status"].isin(["A", "B"])
first_negative = tc[tc["is_negative_balance"]].groupby("account_id")["date"].min().rename("first_negative_date")
loan = loan.merge(first_negative, on="account_id", how="left")
loan["account_ever_negative_balance"] = loan["first_negative_date"].notna()
loan_c = loan

# --- card and order ---
card_c = card_link[["card_id", "disp_id", "account_id", "client_id", "type", "issued"]]
order_c = raw["order"].copy()
order_k = order_c["k_symbol"].where(order_c["k_symbol"].astype("object").str.strip().ne(""))
order_c["k_symbol_clean"] = order_k.fillna("NONE")
disp_c = raw["disp"].copy()

print("Age bands of clients at the end of 1998:")
display(client_c["age_band_end_1998"].value_counts(sort=False).rename("clients").to_frame())
print("Loan status labels:")
display(loan_c.groupby(["status", "status_label"]).size().rename("loans").to_frame())

Age bands of clients at the end of 1998:


,clients
age_band_end_1998,
under 18,72
18-24,730
25-34,962
35-44,952
45-54,982
55-64,853
65+,818


Loan status labels:


,,loans
status,status_label,
A,"Finished, paid",203
B,"Finished, unpaid",31
C,"Running, in order",403
D,"Running, in debt",45


## 8. Reconcile before and after
Cleaning adds columns and labels but never removes or changes a raw value, so counts and totals must match exactly.

In [10]:
clean_tables = {
    "account": account_c, "card": card_c, "client": client_c, "disp": disp_c,
    "district": district_c, "loan": loan_c, "order": order_c, "trans": tc,
}
reconciliation = []


def reconcile(metric, before, after):
    ok = bool(np.isclose(float(before), float(after))) if isinstance(before, (int, float, np.number)) else before == after
    reconciliation.append({"metric": metric, "before": before, "after": after, "match": ok})
    check(f"Reconciliation: {metric}", ok, f"before {before:,}  after {after:,}" if isinstance(before, (int, float, np.number)) else f"{before} vs {after}")


for name, df in clean_tables.items():
    reconcile(f"{name}: row count", baseline["rows"][name], len(df))

after_amount = tc.groupby("type", observed=True)["amount"].sum().to_dict()
after_rows = tc["type"].value_counts().to_dict()
for t_type in sorted(baseline["amount_by_type"]):
    reconcile(f"trans type {t_type}: sum of amount", baseline["amount_by_type"][t_type], after_amount[t_type])
    reconcile(f"trans type {t_type}: row count", baseline["rows_by_type"][t_type], after_rows[t_type])
reconcile("trans: sum of balance", baseline["balance_sum"], float(tc["balance"].sum()))
reconcile("trans: first date", str(baseline["first_date"].date()), str(tc["date"].min().date()))
reconcile("trans: last date", str(baseline["last_date"].date()), str(tc["date"].max().date()))
reconcile("trans: distinct accounts", baseline["n_accounts"], tc["account_id"].nunique())
reconcile("trans: sum of signed amount equals credits minus debits", float(tc["signed_amount"].sum()), float(after_amount.get("PRIJEM", 0) - after_amount.get("VYDAJ", 0) - after_amount.get("VYBER", 0)))
reconcile("trans: unchanged raw amounts", float(trans["amount"].sum()), float(tc["amount"].sum()))

reconciliation = pd.DataFrame(reconciliation)
reconciliation

[PASS] Reconciliation: account: row count  (before 4,500  after 4,500)
[PASS] Reconciliation: card: row count  (before 892  after 892)
[PASS] Reconciliation: client: row count  (before 5,369  after 5,369)
[PASS] Reconciliation: disp: row count  (before 5,369  after 5,369)
[PASS] Reconciliation: district: row count  (before 77  after 77)
[PASS] Reconciliation: loan: row count  (before 682  after 682)
[PASS] Reconciliation: order: row count  (before 6,471  after 6,471)
[PASS] Reconciliation: trans: row count  (before 1,056,320  after 1,056,320)
[PASS] Reconciliation: trans type PRIJEM: sum of amount  (before 3,227,482,664  after 3,227,482,664)
[PASS] Reconciliation: trans type PRIJEM: row count  (before 405,083  after 405,083)
[PASS] Reconciliation: trans type VYBER: sum of amount  (before 208,603,758  after 208,603,758)
[PASS] Reconciliation: trans type VYBER: row count  (before 16,666  after 16,666)
[PASS] Reconciliation: trans type VYDAJ: sum of amount  (before 2,821,775,775  after 2,

,metric,before,after,match
0,account: row count,4500,4500,True
1,card: row count,892,892,True
2,client: row count,5369,5369,True
3,disp: row count,5369,5369,True
4,district: row count,77,77,True
5,loan: row count,682,682,True
6,order: row count,6471,6471,True
7,trans: row count,1056320,1056320,True
8,trans type PRIJEM: sum of amount,3227482664,3227482664,True
9,trans type PRIJEM: row count,405083,405083,True


In [11]:
# Keys, nulls and relationships in the clean tables
KEYS = {"account": "account_id", "card": "card_id", "client": "client_id", "disp": "disp_id", "district": "district_id", "loan": "loan_id", "order": "order_id", "trans": "trans_id"}
for name, key in KEYS.items():
    check(f"{name}_clean: {key} is unique and never null", clean_tables[name][key].is_unique and clean_tables[name][key].notna().all())

must_not_be_null = ["trans_id", "account_id", "date", "type", "amount", "balance", "signed_amount", "balance_before", "transaction_category", "balance_check"]
nulls = tc[must_not_be_null].isna().sum()
check("trans_clean: key and derived columns have no nulls", (nulls == 0).all(), str(nulls[nulls > 0].to_dict()))

for child, col, parent in [
    ("account", "district_id", "district"), ("client", "district_id", "district"), ("disp", "client_id", "client"),
    ("disp", "account_id", "account"), ("card", "disp_id", "disp"), ("loan", "account_id", "account"),
    ("order", "account_id", "account"), ("trans", "account_id", "account"),
]:
    unmatched = int((~clean_tables[child][col].isin(clean_tables[parent][KEYS[parent]])).sum())
    check(f"{child}_clean.{col} -> {parent}: no orphan rows", unmatched == 0, f"{unmatched:,} unmatched")
check("Every account has an owner with a known age band", account_c["owner_age_band"].notna().all())

[PASS] account_clean: account_id is unique and never null
[PASS] card_clean: card_id is unique and never null
[PASS] client_clean: client_id is unique and never null
[PASS] disp_clean: disp_id is unique and never null
[PASS] district_clean: district_id is unique and never null
[PASS] loan_clean: loan_id is unique and never null
[PASS] order_clean: order_id is unique and never null
[PASS] trans_clean: trans_id is unique and never null
[PASS] trans_clean: key and derived columns have no nulls  ({})
[PASS] account_clean.district_id -> district: no orphan rows  (0 unmatched)
[PASS] client_clean.district_id -> district: no orphan rows  (0 unmatched)
[PASS] disp_clean.client_id -> client: no orphan rows  (0 unmatched)
[PASS] disp_clean.account_id -> account: no orphan rows  (0 unmatched)
[PASS] card_clean.disp_id -> disp: no orphan rows  (0 unmatched)
[PASS] loan_clean.account_id -> account: no orphan rows  (0 unmatched)
[PASS] order_clean.account_id -> account: no orphan rows  (0 unmatched)

In [12]:
# Regression checks against what profiling found (only meaningful on the full dataset)
if FULL:
    found = {
        "zero-amount rows": (int(tc["is_zero_amount"].sum()), 162),
        "negative-balance rows": (int(tc["is_negative_balance"].sum()), 2999),
        "blank operation rows": (int((tc["operation_clean"] == "NONE").sum()), 183114),
        "k_symbol missing": (int((tc["k_symbol_kind"] == "missing").sum()), 481881),
        "k_symbol single space": (int((tc["k_symbol_kind"] == "space").sum()), 53433),
        "partner account was zero": (int(tc["partner_account_was_zero"].sum()), 21881),
        "rows with partner bank and account": (int(has_counterparty.sum()), 273508),
        "identical-looking rows": (len(repeated), 3448),
        "identical-looking groups": (n_repeated_groups, 1723),
        "outgoing transfers without a counterparty": (len(no_counterparty_transfer), 1),
        "loans with status A": (int((loan_c["status"] == "A").sum()), 203),
        "loans with status B": (int((loan_c["status"] == "B").sum()), 31),
        "loans with status C": (int((loan_c["status"] == "C").sum()), 403),
        "loans with status D": (int((loan_c["status"] == "D").sum()), 45),
        "accounts that ever had a negative balance": (int(tc.loc[tc["is_negative_balance"], "account_id"].nunique()), 288),
    }
    for label, (got, expected) in found.items():
        check(f"Matches profiling: {label}", got == expected, f"got {got:,}, profiling found {expected:,}")
else:
    print("Not the full dataset: regression checks against the profiling results are skipped.")

[PASS] Matches profiling: zero-amount rows  (got 162, profiling found 162)
[PASS] Matches profiling: negative-balance rows  (got 2,999, profiling found 2,999)
[PASS] Matches profiling: blank operation rows  (got 183,114, profiling found 183,114)
[PASS] Matches profiling: k_symbol missing  (got 481,881, profiling found 481,881)
[PASS] Matches profiling: k_symbol single space  (got 53,433, profiling found 53,433)
[PASS] Matches profiling: partner account was zero  (got 21,881, profiling found 21,881)
[PASS] Matches profiling: rows with partner bank and account  (got 273,508, profiling found 273,508)
[PASS] Matches profiling: identical-looking rows  (got 3,448, profiling found 3,448)
[PASS] Matches profiling: identical-looking groups  (got 1,723, profiling found 1,723)
[PASS] Matches profiling: outgoing transfers without a counterparty  (got 1, profiling found 1)
[PASS] Matches profiling: loans with status A  (got 203, profiling found 203)
[PASS] Matches profiling: loans with status B  (g

## 9. Data quality report and exceptions

In [13]:
total = len(tc)
neg_accounts = tc.loc[tc["is_negative_balance"], "account_id"].nunique()
log_issue("trans", "operation", "Blank operation (exactly the interest credits)", (tc["operation_clean"] == "NONE").sum(), total, "Kept; operation_clean = NONE; category 'Interest credited'")
log_issue("trans", "k_symbol", "Missing k_symbol", (tc["k_symbol_kind"] == "missing").sum(), total, "Kept; k_symbol_clean = NONE; k_symbol_kind = missing")
log_issue("trans", "k_symbol", "Single space instead of a k_symbol (only on debits)", (tc["k_symbol_kind"] == "space").sum(), total, "Kept; k_symbol_clean = NONE; k_symbol_kind = space")
log_issue("trans", "bank", "No partner bank", tc["partner_bank"].isna().sum(), total, "Kept as NULL (no counterparty)")
log_issue("trans", "account", "Partner account is 0 (only on cash and card withdrawals)", tc["partner_account_was_zero"].sum(), total, "partner_account = NULL; partner_account_was_zero = 1")
log_issue("trans", "account", "No partner account and no zero", (tc["partner_account"].isna() & ~tc["partner_account_was_zero"]).sum(), total, "Kept as NULL (no counterparty)")
log_issue("trans", "amount", "Zero-amount transactions (interest postings)", tc["is_zero_amount"].sum(), total, "Kept; is_zero_amount = 1")
log_issue("trans", "balance", f"Negative balance (in {neg_accounts:,} accounts)", tc["is_negative_balance"].sum(), total, "Kept; is_negative_balance = 1")
log_issue("trans", "(all but trans_id)", "Identical-looking rows (kept: most are separate postings)", len(repeated), total, "Kept; is_repeated_posting = 1; repeat_group_size")
log_issue("trans", "(all but trans_id)", "Identical-looking rows whose members share one balance (possible phantom copies)", tc["possible_phantom_copy"].sum(), total, "Kept; possible_phantom_copy = 1; listed in dq_exceptions.csv")
log_issue("trans", "balance", "Account-day balance differs by up to 2 from the previous day (rounding of whole-number values)", (tc["balance_check"] == "rounding").sum(), total, "Kept; balance_check = rounding")
log_issue("trans", "balance", "Account-day balance differs by more than 2 from the previous day", (tc["balance_check"] == "review").sum(), total, "Kept; balance_check = review; listed in dq_exceptions.csv")
log_issue("trans", "(order)", "Rows on account-days with several postings (order within the day unknown; trans_id is not chronological)", (tc["account_day_rows"] > 1).sum(), total, "No order derived; balance_before = balance - signed_amount per row")
log_issue("trans", "operation/bank/account", "Outgoing transfer with no counterparty", len(no_counterparty_transfer), total, "Kept; listed in dq_exceptions.csv")
log_issue("district", "A12 (unemployment_rate_1995)", "Missing value", district_c["unemployment_rate_1995"].isna().sum(), len(district_c), "Kept as NULL")
log_issue("district", "A15 (crimes_1995)", "Missing value", district_c["crimes_1995"].isna().sum(), len(district_c), "Kept as NULL")
log_issue("loan", "status", "Status is fully determined by 'balance ever negative' and 'end date reaches 1999'", len(loan_c), len(loan_c), "Kept with labels; no predictive claims")
log_issue("client", "birth_date", "Exact date of birth is personal data", len(client_c), len(client_c), "Not exported; birth_year, age and age band only")

dq_report = pd.DataFrame(dq)
display(dq_report)

add_exceptions("outgoing transfer without a counterparty", "trans", no_counterparty_transfer["trans_id"], "operation = PREVOD NA UCET but no partner bank or account")
add_exceptions("possible phantom copy", "trans", tc.loc[tc["possible_phantom_copy"], "trans_id"], "identical rows with the same balance and amount > 0")
exceptions = pd.concat(exception_frames, ignore_index=True) if exception_frames else pd.DataFrame(columns=["issue", "table", "key", "note"])
print(f"Exceptions listed: {len(exceptions):,}")
display(exceptions.groupby("issue").size().rename("rows").to_frame())

,table,column,issue,rows_affected,share_pct,handling
0,trans,operation,Blank operation (exactly the interest credits),183114,17.34,Kept; operation_clean = NONE; category 'Intere...
1,trans,k_symbol,Missing k_symbol,481881,45.62,Kept; k_symbol_clean = NONE; k_symbol_kind = m...
2,trans,k_symbol,Single space instead of a k_symbol (only on de...,53433,5.06,Kept; k_symbol_clean = NONE; k_symbol_kind = s...
3,trans,bank,No partner bank,782812,74.11,Kept as NULL (no counterparty)
4,trans,account,Partner account is 0 (only on cash and card wi...,21881,2.07,partner_account = NULL; partner_account_was_ze...
5,trans,account,No partner account and no zero,760931,72.04,Kept as NULL (no counterparty)
6,trans,amount,Zero-amount transactions (interest postings),162,0.02,Kept; is_zero_amount = 1
7,trans,balance,Negative balance (in 288 accounts),2999,0.28,Kept; is_negative_balance = 1
8,trans,(all but trans_id),Identical-looking rows (kept: most are separat...,3448,0.33,Kept; is_repeated_posting = 1; repeat_group_size
9,trans,(all but trans_id),Identical-looking rows whose members share one...,8,0.00,Kept; possible_phantom_copy = 1; listed in dq_...


Exceptions listed: 10


,rows
issue,
balance_check = review,1
outgoing transfer without a counterparty,1
possible phantom copy,8


## 10. Export
Flags are written as 0/1 so MySQL and Power BI read them as numbers. Empty cells mean NULL.

In [14]:
FLAGS = [
    "is_zero_amount", "is_negative_balance", "partner_account_was_zero", "is_repeated_posting", "possible_phantom_copy",
]
TRANS_EXPORT = [
    "trans_id", "account_id", "date", "year", "month", "year_month", "month_start",
    "type", "type_label", "direction", "operation", "operation_clean", "k_symbol", "k_symbol_clean", "k_symbol_kind",
    "transaction_category", "amount", "signed_amount", "balance", "balance_before",
    "partner_bank", "partner_account", "partner_account_was_zero",
    "is_zero_amount", "is_negative_balance", "is_repeated_posting", "repeat_group_size", "possible_phantom_copy",
    "account_day_rows", "account_day_residual", "balance_check",
]
trans_export = tc[TRANS_EXPORT].copy()
for col in FLAGS:
    trans_export[col] = trans_export[col].astype("int8")
trans_export["month_start"] = trans_export["month_start"].dt.strftime("%Y-%m-%d")

account_export = account_c.copy()
for col in ["has_loan", "has_card"]:
    account_export[col] = account_export[col].astype("int8")
loan_export = loan_c.copy()
for col in ["is_finished", "account_ever_negative_balance"]:
    loan_export[col] = loan_export[col].astype("int8")

exports = {
    "trans_clean": trans_export,
    "account_clean": account_export,
    "client_clean": client_c,
    "disp_clean": disp_c,
    "district_clean": district_c,
    "loan_clean": loan_export,
    "card_clean": card_c,
    "order_clean": order_c,
    "dq_exceptions": exceptions,
}
PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(exist_ok=True)
for name, df in exports.items():
    df.to_csv(PROCESSED / f"{name}.csv", index=False)

# Reports that contain only counts and totals (safe to commit)
dq_report.to_csv(REPORTS / "data_quality_report.csv", index=False)
reconciliation.to_csv(REPORTS / "cleaning_reconciliation.csv", index=False)

md_lines = ["# Data quality report", "", "| Table | Column | Issue | Rows | Share % | Handling |", "|---|---|---|---:|---:|---|"]
for r in dq_report.itertuples(index=False):
    md_lines.append(f"| {r.table} | {r.column} | {r.issue} | {r.rows_affected:,} | {r.share_pct} | {r.handling} |")
(REPORTS / "data_quality_report.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")

sizes = pd.Series({f"{name}.csv": (PROCESSED / f"{name}.csv").stat().st_size / 1e6 for name in exports}, name="size_MB")
display(sizes.to_frame())

,size_MB
trans_clean.csv,177.05
account_clean.csv,0.32
client_clean.csv,0.13
disp_clean.csv,0.12
district_clean.csv,0.01
loan_clean.csv,0.05
card_clean.csv,0.03
order_clean.csv,0.26
dq_exceptions.csv,0.00


## 11. Check log

In [15]:
log = pd.DataFrame(checks)
log.to_csv(REPORTS / "cleaning_checks.csv", index=False)
print(log["status"].value_counts().to_string())
display(log[log["status"] != "PASS"])

failed = log[log["status"] == "FAIL"]
assert failed.empty, f"{len(failed)} check(s) failed:\n{failed[['check', 'detail']].to_string(index=False)}"
print("\nAll checks passed. Clean files are in data/processed/ (not committed).")

status
PASS    56


,check,status,detail



All checks passed. Clean files are in data/processed/ (not committed).


## 12. Your notes (write these in your own words)
1. For each profiling finding, which decision did we take and why is it safe (nothing deleted, totals reconcile)?
2. Why do we keep the identical-looking rows, and which rows are flagged for review instead?
3. Why does `balance_before = balance - signed_amount` avoid the within-day ordering problem?
4. What should you never claim from the loan status columns, and why?